# Predicted Epitopes of Trypanosoma cruzi based on Phage Display Data Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install --quiet mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset URL -- always use this variable throughout
croissant_url = 'https://sen.science/doi/10.71728/senscience.etbd-dths/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# View the Croissant schema metadata
meta_obj = dataset.metadata  # This is a wrapper object, not a dict; let's print some key fields safely
print(f"Name: {getattr(meta_obj, 'name', None)}")
print(f"Identifier: {getattr(meta_obj, 'identifier', None)}")
print(f"Version: {getattr(meta_obj, 'version', None)}")

print("\nDescription:\n", getattr(meta_obj, 'description', None))

## 2. Data Overview
Review available record sets, fields, and their IDs.

For every entity in mlcroissant, you should reference objects by their `@id`, never just by name. The structure returned by `dataset.metadata` describes the available record sets and fields we can access in this dataset.

In [ ]:
# List available record sets by @id
print("Available record sets (by @id):")
for rs in dataset.record_sets():
    print(f"  @id: {rs.id} | name: {getattr(rs, 'name', '')}")

print("\nExample fields for each record set:")
for rs in dataset.record_sets():
    print(f"\nRecordSet @id: {rs.id}")
    for field in rs.fields:
        print(f"    Field @id: {field.id} | name: {getattr(field, 'name', '')} | dataType: {getattr(field, 'data_type', '')}")

## 3. Data Extraction
Load data from specific record sets (@id) into DataFrames. Use the record set and field `@id`s you listed above: this allows dynamic extraction, robust to name changes.

*Tip: Some record sets may be large -- you can set `max_records` in `dataset.records()` if you want a preview only.*

In [ ]:
# Let's collect all record set @ids
record_set_ids = [rs.id for rs in dataset.record_sets()]
print(f"Detected {len(record_set_ids)} record sets:")
for rsid in record_set_ids:
    print(f" - {rsid}")

dataframes = {}
for record_set_id in record_set_ids:
    print(f"\nLoading records from {record_set_id} ...")
    records = list(dataset.records(record_set=record_set_id))
    # Defensive: check if records are present
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"  Loaded {len(df)} records, columns: {df.columns.tolist()}")
    else:
        print("  No records found.")

# For illustration, print example DataFrame (first available)
if dataframes:
    example_rs = list(dataframes.keys())[0]
    print(f"\nExample DataFrame for record set @id: {example_rs}")
    display(dataframes[example_rs].head())

## 4. Exploratory Data Analysis (EDA)
Now, let's perform basic EDA: filtering, normalizing, and grouping.

1. Pick a record set and a numeric field by their `@id`.
2. Filter records above a threshold.
3. Normalize the selected numeric field (z-score).
4. Group by a categorical field (if present) and summarize.

In [ ]:
# We'll use the first non-empty DataFrame and pick fields by @id, as required.
if dataframes:
    record_set_id = example_rs
    df = dataframes[record_set_id]
    print(f"Fields of record set {record_set_id}: {df.columns.tolist()}")
    # Try to automatically detect a numeric field (float or int)
    possibles = df.select_dtypes('number').columns.tolist()
    if not possibles:
        print("No numeric fields available for EDA.")
    else:
        numeric_field = possibles[0]
        print(f"Using {numeric_field} as numeric field.")
        threshold = df[numeric_field].mean()  # Use mean as a simple threshold
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered to {len(filtered_df)} records with {numeric_field} > mean ({threshold:.3f})")
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try to group by the first categorical field
        cat_fields = [col for col in df.columns if df[col].dtype == 'object']
        group_field = cat_fields[0] if cat_fields else None
        if group_field:
            grouped = filtered_df.groupby(group_field)[numeric_field].mean()
            print(f"\nGrouped mean {numeric_field} by '{group_field}' (first 5 groups):")
            print(grouped.head())
        else:
            print("No categorical field found for grouping.")
else:
    print("No tabular data available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Here, we use matplotlib and seaborn for summary visualization.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Basic distribution plot for the numeric field
if dataframes and possibles:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field} in {record_set_id}")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    # If we have a group field, plot a boxplot as well
    if group_field:
        plt.figure(figsize=(12,5))
        sns.boxplot(y=group_field, x=numeric_field, data=df, orient='h')
        plt.title(f"{numeric_field} by {group_field}")
        plt.show()

## 6. Conclusion
In this notebook, you learned how to explore a Croissant-formatted dataset using the `mlcroissant` library. We programmatically discovered record sets, fields, performed basic EDA, and visualized data distributions --- referencing all entities strictly by their `@id` fields.

This workflow is robust for FAIR data structures and can be adapted for similar datasets in the future.